In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import time # calculate elapased time

In [3]:
attributes = ['year_of_birth', 'name', 'assigned_sex_at_birth', 'count']
# sample_data = {'year_of_birth': '1880', 'name': 'vee', 'assigned_sex_at_birth':"M", 'count':'500'}
df = pd.DataFrame(columns=attributes)
print(df)

Empty DataFrame
Columns: [year_of_birth, name, assigned_sex_at_birth, count]
Index: []


In [4]:
def print_output(content):
    with open("output.txt", "a") as file:
        file.write(content)

In [5]:
def extract(file, df):
    f = open(file, 'r')
    year = file[-8:-4]
    for line in f:
        stripped_line = line.strip().split(',')
        # Using a dict
        data_dict = {'year_of_birth': year,
                     'name': stripped_line[0],
                     'assigned_sex_at_birth':stripped_line[1],
                     'count':stripped_line[2]
        }
        
        temp_df = pd.DataFrame(data_dict, index=[0])
        df = pd.concat([df,temp_df], ignore_index=True)
    return df

#To process 9 files. It takes 7.379 minutes or 442.74 seconds.. Way too slow
# 1 file process = Total time: 0.8409976959228516
# Referencing code from function @extract_data() - https://github.com/ctrlvee/Extract_GDP/blob/main/etl_project_gdp.py

In [6]:
import fileinput

In [7]:
def extract2(file, df):
    year = file[-8:-4]
    with fileinput.input(files=file, encoding="utf-8") as f:
        for line in f:
            stripped_line = line.strip().split(',')
            # Using a dict
            data_dict = {'year_of_birth': year,
                         'name': stripped_line[0],
                         'assigned_sex_at_birth':stripped_line[1],
                         'count':stripped_line[2]
            }
            
            temp_df = pd.DataFrame(data_dict, index=[0])
            df = pd.concat([df,temp_df], ignore_index=True)
    return df

#To process 9 files. It takes 8 minutes or 476.12 seconds.. Way too slow
# 1 file process = Total time: 0.7093737125396729

In [8]:
# Extract and convert each file to a CSV and then a Pandas

def extract3(file,df):
    year = file[-8:-4]
    data = pd.read_csv(file, header=None)
    data.columns = ['name', 'assigned_sex_at_birth','count']
    # Create and Insert new column at start
    data.insert(0, 'year_of_birth', year)
    df = pd.concat([df,data], ignore_index=True)

# To process 9 files. It takes Total time: 0.5635101795196533
# To process 1 file = Total time: 0.004897117614746094 

a = (0.56351-476.12)/(476.12)*100
print(a)

# This solution is 99.8816% faster 

-99.88164538351676


# Extract Testing
> Problem: I think my extract function is only reading one line per file
>

In [9]:
test_file = '/kaggle/input/baby-names-year-1880-2023-social-security-admin/names/yob1880.txt'
copy_df_one = df.copy()
copy_df_two = df.copy()
copy_df_three = df.copy()

def test_extract_one(test_file, df):
    start = time.time()
    df = extract2(test_file, df)
    end = time.time()
    print(f'Extract Function One Total time: {end-start}')

def test_extract_two(test_file, df):
    start = time.time()
    df = extract2(test_file, df)
    end = time.time()
    print(f'Extract Function Two Total time: {end-start}')

def test_extract_three(test_file, df):
    start = time.time()
    df = extract3(test_file, df)
    end = time.time()
    print(f'Extract Function Three Total time: {end-start}')


print("Testing each extract function with one test file and posting their processing times")
test_extract_one(test_file, copy_df_one)
test_extract_two(test_file, copy_df_two)
test_extract_three(test_file, copy_df_three)

Testing each extract function with one test file and posting their processing times
Extract Function One Total time: 0.85335373878479
Extract Function Two Total time: 0.7448983192443848
Extract Function Three Total time: 0.01379251480102539


# Test Implementation to all files
> Uncomment to try with 9 test files

In [10]:
# # Measure elapsed time 
# #Save timestamp
# start = time.time()

# count = 0
# import os 
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         if filename != 'NationalReadMe.pdf':
#             if count != 10:
#                 count += 1
#                 df = extract3(os.path.join(dirname,filename), df)
#             else:
#                 pass
#     print(count)

# end = time.time()
# print(f'Total time: {end-start}')

# Change so it shows a raw df and timestamp from each extract function 


In [11]:
# Measure elapsed time 
start = time.time()

count = 0
import os 
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename != 'NationalReadMe.pdf':
                df = extract3(os.path.join(dirname,filename), df)
                count+=1


print(count)
end = time.time()
print(f'Total time: {end-start}')


144
Total time: 3.3183329105377197


In [12]:
# print(df.size)
# print("columns " + df.columns)
# print(df.head())

In [13]:
# Parse through each file
# Ignore initial file
# Add to a pandas dataframe 
# Create a column with year so it's organized and easy to measure
# Analyze
# how many Ferdinand's per year 
# what year peaked
# What year had least


# Errors I got + Resources that helped me debug
> TypeError: first argument must be an iterable of pandas objects, you passed an object
           of type "DataFrame"
* https://www.statology.org/typeerror-first-argument-must-be-an-iterable-of-pandas-objects/
> ValueError: Shape of passed values is (4, 1), indices imply (4, 4)
* https://stackoverflow.com/questions/50874117/pandas-dataframe-shape-of-passed-values-is-1-4-indices-imply-4-4/50874193
* https://pandas.pydata.org/pandas-docs/version/1.4/reference/api/pandas.DataFrame.append.html
> ValueError: If using all scalar values, you must pass an index' When Merging Multiple DataFrames
* https://saturncloud.io/blog/resolving-valueerror-if-using-all-scalar-values-you-must-pass-an-index-when-merging-multiple-dataframes/#:~:text=understand%20the%20error.-,The%20%22ValueError%3A%20If%20using%20all%20scalar%20values%2C%20you%20must,integers%2C%20floats%2C%20or%20strings.
* 

# Concerns
> How can I optimize my Extract function?
* I am currently using a dictionary and appending for each LINE so processing will take wayyyy longer and is relative to the size of EACH File
* What is the Big(O) Notation for my current solution? (https://stackoverflow.com/questions/3255/big-o-how-do-you-calculate-approximate-it)
* To process 144 text files with thousands of rows each, it is going to take A LONG time.